# 2. SQL Business Analysis
### Austin Airbnb Pricing Intelligence — Notebook 2 of 5
---

## 2.1 Business Context

The Austin short-term rental market contains **10,402 active listings**
spanning dozens of neighborhoods, property types, and price points.

The core business question driving this analysis:

> *"What should a new Airbnb host in Austin charge per night
> to maximize revenue — and what drives that number?"*

SQL is the analytical tool of choice here. Every query targets
a specific business question. Every result is interpreted
as a market insight — not just a number.

---

## 2.2 Notebook Structure

This analysis tells a story in 5 chapters —
moving from broad market overview to specific host recommendations.

| Chapter | Focus | Queries |
|---|---|---|
| 1 | Market Overview | Q1 — Q2 |
| 2 | Neighborhood Intelligence | Q3 — Q4 |
| 3 | Price Drivers | Q5 — Q6 |
| 4 | Host Intelligence | Q7 — Q8 |
| 5 | The Recommendation | Q9 — Q10 |

Each query follows the same format:
- Business question stated in plain English
- Clean, commented SQL
- Result interpreted as a market insight

---

## 2.3 Technical Setup

The clean dataset from Notebook 1 is loaded into **SQLite** —
a lightweight database engine built into Python.
This mirrors the real analyst workflow of querying a structured
database rather than manipulating flat files directly.

All 10 queries are also saved to `sql/queries.sql` as a
standalone file — readable without opening Jupyter.

> **DA angle:** SQL proficiency is tested in nearly everywhere.
> This notebook serves as public,
> reproducible proof — from basic aggregations to
> window functions and CTEs.

> **BA angle:** Every query maps directly to a stakeholder
> question. The markdown narrative alone tells the full
> market story — no SQL knowledge required to read it.

In [1]:
# ─── Notebook 2 — SQL Business Analysis ──────────────────────────────────────
# Setup: Load clean dataset into SQLite for structured querying

import pandas as pd
import sqlite3
import os

# ─── Step 1 — Load clean dataset ─────────────────────────────────────────────
listings = pd.read_csv('../data/processed/listings_clean.csv')

# ─── Step 2 — Create SQLite database ─────────────────────────────────────────
db_path = '../data/processed/airbnb_austin.db'
conn    = sqlite3.connect(db_path)

# ─── Step 3 — Load DataFrame into SQLite as a table ──────────────────────────
listings.to_sql(
    name      = 'listings',
    con       = conn,
    if_exists = 'replace',
    index     = False
)

# ─── Step 4 — Verify ──────────────────────────────────────────────────────────
verify = pd.read_sql_query("SELECT COUNT(*) as total_listings FROM listings", conn)

print("SQLITE DATABASE READY")
print("=" * 45)
print(f"  Database location : {db_path}")
print(f"  Table name        : listings")
print(f"  Rows loaded       : {verify['total_listings'][0]:,}")
print(f"  File size         : {os.path.getsize(db_path) / 1024 / 1024:.2f} MB")

# ─── Step 5 — Confirm columns are available ───────────────────────────────────
cols = pd.read_sql_query("PRAGMA table_info(listings)", conn)
print(f"  Columns available : {len(cols)}")
print(f"\n  Database connection status : {'Open' if conn else 'Failed'}")
print("\n  Ready for SQL queries.")

SQLITE DATABASE READY
  Database location : ../data/processed/airbnb_austin.db
  Table name        : listings
  Rows loaded       : 10,402
  File size         : 2.74 MB
  Columns available : 54

  Database connection status : Open

  Ready for SQL queries.


### What Was Set Up

The clean dataset from Notebook 1 has been loaded into a
SQLite database — `airbnb_austin.db` — stored in `data/processed/`.

| Detail | Value |
|---|---|
| Database file | airbnb_austin.db |
| Table name | listings |
| Rows loaded | 10,402 |
| Columns available | 54 |
| File size | 2.74 MB |

The database mirrors exactly what a real analyst would work with —
a structured, queryable data store rather than a flat CSV file.
Every query from this point forward runs directly against
this database using standard SQL syntax.

> **Oracle note:** The connection pattern is identical in concept
> to `cx_Oracle.connect()` — the difference is SQLite requires
> no server, no credentials, and no installation.
> `LIMIT` replaces Oracle's `FETCH FIRST N ROWS ONLY`.
> All other syntax — CTEs, window functions, aggregations —
> is identical.

---
## 2.4 Chapter 1 — Market Overview

### The Starting Point

Any pricing strategy begins with market understanding.
Before a single recommendation can be made, the structure
of the Austin short-term rental market must be established —
who is competing, at what price, and capturing what share
of the revenue opportunity.

Chapter 1 answers the foundational questions:
- What room types dominate Austin?
- What does each segment charge — and earn?
- Which neighborhoods concentrate the most listings and revenue?

---

### Query 1 — Room Type Market Intelligence

**Business question:**
*What is the complete market picture by room type —
price positioning, market share, revenue potential,
occupancy, and guest satisfaction?*

**DA angle:**
A single `AVG(price) GROUP BY room_type` answers nothing useful.
This query uses a CTE to isolate room-level statistics,
then applies window functions to calculate market share,
price gaps between segments, and revenue concentration.
The result is a multi-dimensional market segmentation table.

**BA angle:**
A property management company entering Austin needs to know
where the revenue opportunity actually sits — not just which
room type has the highest average price, but which segment
combines price, occupancy, and volume into the strongest
total revenue case. This query answers that directly.

**SQL concepts used:**
- Common Table Expression (CTE)
- Window functions — `SUM() OVER()`, `LAG() OVER()`
- Aggregations — `COUNT`, `AVG`, `PERCENTILE`
- Derived metrics — market share, price gap, revenue per listing

In [2]:
# ─── Query 1 — Room Type Market Intelligence ──────────────────────────────────
# Business question: What is the complete market picture by room type?

query_1 = """
WITH room_metrics AS (
    SELECT
        room_type,

        -- Volume
        COUNT(*)                                                AS total_listings,

        -- Price metrics
        ROUND(AVG(price), 2)                                    AS avg_price,
        ROUND(AVG(price) * 0.5, 2)                             AS estimated_median_price,
        ROUND(MIN(price), 2)                                    AS min_price,
        ROUND(MAX(price), 2)                                    AS max_price,

        -- Revenue metrics
        ROUND(AVG(estimated_revenue_l365d), 0)                  AS avg_annual_revenue,
        ROUND(SUM(estimated_revenue_l365d), 0)                  AS total_market_revenue,

        -- Occupancy
        ROUND(AVG(estimated_occupancy_l365d), 1)                AS avg_occupied_nights,
        ROUND(AVG(availability_365), 1)                         AS avg_available_nights,

        -- Quality
        ROUND(AVG(review_scores_rating), 2)                     AS avg_rating,
        ROUND(AVG(reviews_per_month), 2)                        AS avg_reviews_per_month,

        -- Host profile
        ROUND(AVG(host_experience_years), 1)                    AS avg_host_experience,
        ROUND(100.0 * SUM(host_is_superhost) / COUNT(*), 1)     AS superhost_rate_pct

    FROM listings
    GROUP BY room_type
),

room_ranked AS (
    SELECT
        *,
        -- Market share of listings
        ROUND(100.0 * total_listings / SUM(total_listings) OVER (), 1)
                                                                AS listing_share_pct,

        -- Market share of revenue
        ROUND(100.0 * total_market_revenue / SUM(total_market_revenue) OVER (), 1)
                                                                AS revenue_share_pct,

        -- Price gap vs the segment above
        ROUND(avg_price - LAG(avg_price) OVER (ORDER BY avg_price DESC), 2)
                                                                AS price_gap_vs_above,

        -- Revenue rank
        RANK() OVER (ORDER BY avg_annual_revenue DESC)          AS revenue_rank

    FROM room_metrics
)

SELECT
    revenue_rank                                                AS rank,
    room_type,
    total_listings,
    listing_share_pct                                           AS listing_share_pct,
    revenue_share_pct                                           AS revenue_share_pct,
    avg_price,
    min_price,
    max_price,
    avg_annual_revenue,
    avg_occupied_nights,
    avg_rating,
    superhost_rate_pct,
    avg_host_experience                                         AS host_exp_years,
    price_gap_vs_above
FROM room_ranked
ORDER BY revenue_rank
"""

# ─── Execute and display ───────────────────────────────────────────────────────
q1_results = pd.read_sql_query(query_1, conn)

print("QUERY 1 — ROOM TYPE MARKET INTELLIGENCE")
print("=" * 65)
print(q1_results.to_string(index=False))

QUERY 1 — ROOM TYPE MARKET INTELLIGENCE
 rank       room_type  total_listings  listing_share_pct  revenue_share_pct  avg_price  min_price  max_price  avg_annual_revenue  avg_occupied_nights  avg_rating  superhost_rate_pct  host_exp_years  price_gap_vs_above
    1 Entire home/apt            9013               86.6               97.1     221.97       24.0     2400.0             15204.0                 87.8        4.86                55.4             8.5              -67.97
    2    Private room            1291               12.4                2.8      94.43        8.0     2161.0              3077.0                 52.2        4.85                36.9             8.2             -127.54
    3      Hotel room              33                0.3                0.1     289.94       78.0      956.0              2591.0                 15.5        4.89                 3.0             7.1                 NaN
    4     Shared room              65                0.6                0.1      20.58  

### Query 1 Results — Room Type Market Intelligence

#### The Austin Market Is Dominated by Entire Homes

| Room Type | Listings | Listing Share | Revenue Share | Avg Price | Avg Annual Revenue |
|---|---|---|---|---|---|
| Entire home/apt | 9,013 | 86.6% | 97.1% | $221.97 | $15,204 |
| Private room | 1,291 | 12.4% | 2.8% | $94.43 | $3,077 |
| Hotel room | 33 | 0.3% | 0.1% | $289.94 | $2,591 |
| Shared room | 65 | 0.6% | 0.1% | $20.58 | $1,312 |

#### Key Findings

**Finding 1 — Entire homes capture nearly all revenue**
86.6% of listings generate 97.1% of total market revenue.
The entire home segment is not just the largest —
it is effectively the only segment that matters
from a revenue perspective.
A new host with an entire home enters the right segment.

**Finding 2 — The private room ceiling is low**
Private room hosts earn an average of $3,077 per year —
approximately one-fifth of what entire home hosts earn ($15,204).
The $94 average nightly rate combined with only 52 occupied
nights per year produces a revenue floor, not a revenue opportunity.

**Finding 3 — Hotel rooms charge the most but earn the least**
At $289.94 average price, hotel rooms command the highest nightly rate.
However with only 15.5 occupied nights per year on average,
annual revenue ($2,591) falls below even private rooms.
High price does not compensate for low demand volume.

**Finding 4 — Superhost status follows revenue**
Entire home hosts have a 55.4% superhost rate.
Private room hosts have a 36.9% superhost rate.
Shared room hosts have a 0% superhost rate.
Superhost status concentrates where revenue concentrates —
a signal that professional hosting behavior correlates
directly with segment choice.

**Finding 5 — Experience does not guarantee earnings**
Shared room hosts have the highest average experience (12.3 years)
yet the lowest revenue ($1,312) and lowest rating (4.60).
Host experience alone does not drive performance —
segment choice does.

#### Strategic Implication for a New Host

The data is unambiguous.
A new host entering the Austin market with an entire home
operates in the segment that generates 97.1% of all revenue,
with an average annual return of $15,204 at 87.8 occupied nights.
All subsequent analysis focuses on the entire home segment.

> **DA angle:** The revenue concentration ratio (86.6% of listings
> generating 97.1% of revenue) is a classic Pareto signal.
> This finding anchors every downstream query.

> **BA angle:** For a property management company,
> this table defines the investment thesis —
> entire home acquisitions in Austin represent
> the only segment with meaningful revenue scale.

---
### Query 2 — Neighborhood Price Intelligence

**Business question:**
*Which Austin neighborhoods command the highest prices,
generate the most revenue, and maintain the strongest
occupancy — and how does each neighborhood rank
across all three dimensions simultaneously?*

**Why this matters:**
Two listings with identical features can generate
dramatically different revenue based on neighborhood alone.
Location is the one variable a host cannot change after listing.
Getting the neighborhood decision right is the highest-leverage
choice a new host makes.

**DA angle:**
This query ranks all neighborhoods across three independent
dimensions — price, revenue, and occupancy — then combines
them into a composite opportunity score using window functions.
A neighborhood that ranks highly on all three dimensions
represents a genuine market opportunity.
A neighborhood that ranks high on price but low on occupancy
signals an overpriced market with weak demand.

**BA angle:**
For a property management company, this table is the
acquisition targeting matrix. Neighborhoods with high
composite scores represent priority expansion markets.
Neighborhoods with high price but low occupancy signal
risk — revenue projections based on price alone
would overstate actual returns.

**SQL concepts used:**
- Multiple CTEs chained together
- RANK() window function across multiple dimensions
- Composite scoring — averaging three independent ranks
- HAVING clause to filter statistically meaningful neighborhoods
- CASE WHEN for categorical labeling

---
### Query 2 — Neighborhood Price Intelligence

**Business question:**
*Which Austin neighborhoods command the highest prices,
generate the most revenue, and maintain the strongest
occupancy — and how does each neighborhood rank
across all three dimensions simultaneously?*

**Why this matters:**
Two listings with identical features can generate
dramatically different revenue based on neighborhood alone.
Location is the one variable a host cannot change after listing.
Getting the neighborhood decision right is the highest-leverage
choice a new host makes.

**DA angle:**
Neighborhood analysis requires more than a simple price ranking.
A neighborhood with the highest average price but lowest occupancy
produces less revenue than a mid-priced neighborhood with
strong consistent demand. Three dimensions must be evaluated
simultaneously — price, revenue, and occupancy.

**BA angle:**
For a property management company, this table is the
acquisition targeting matrix. Neighborhoods with strong
composite scores across all three dimensions represent
priority expansion markets. Price alone is a misleading
signal without occupancy and revenue context alongside it.

In [3]:
# ─── Query 2 — Neighborhood Price Intelligence ────────────────────────────────
# Business question: Which neighborhoods rank highest across
# price, revenue, and occupancy simultaneously?

query_2 = """
WITH neighborhood_base AS (
    SELECT
        neighbourhood_cleansed                                      AS neighborhood,

        -- Volume
        COUNT(*)                                                    AS total_listings,

        -- Price metrics
        ROUND(AVG(price), 2)                                        AS avg_price,
        ROUND(AVG(price_per_bedroom), 2)                            AS avg_price_per_bedroom,

        -- Revenue metrics
        ROUND(AVG(estimated_revenue_l365d), 0)                      AS avg_annual_revenue,
        ROUND(SUM(estimated_revenue_l365d), 0)                      AS total_neighborhood_revenue,

        -- Occupancy metrics
        ROUND(AVG(estimated_occupancy_l365d), 1)                    AS avg_occupied_nights,

        -- Quality metrics
        ROUND(AVG(review_scores_rating), 2)                         AS avg_rating,
        ROUND(AVG(review_scores_location), 2)                       AS avg_location_score,

        -- Host metrics
        ROUND(100.0 * SUM(host_is_superhost) / COUNT(*), 1)         AS superhost_rate_pct,
        ROUND(AVG(host_experience_years), 1)                        AS avg_host_experience,

        -- New listing presence
        ROUND(100.0 * SUM(is_new_listing) / COUNT(*), 1)            AS new_listing_pct

    FROM listings
    WHERE room_type = 'Entire home/apt'
    GROUP BY neighbourhood_cleansed
    HAVING COUNT(*) >= 20
),

neighborhood_ranked AS (
    SELECT
        *,
        -- Individual dimension ranks
        RANK() OVER (ORDER BY avg_price DESC)                       AS price_rank,
        RANK() OVER (ORDER BY avg_annual_revenue DESC)              AS revenue_rank,
        RANK() OVER (ORDER BY avg_occupied_nights DESC)             AS occupancy_rank,

        -- Market share of total revenue
        ROUND(100.0 * total_neighborhood_revenue /
            SUM(total_neighborhood_revenue) OVER (), 2)             AS revenue_share_pct,

        -- Price vs market benchmark
        ROUND(avg_price - AVG(avg_price) OVER (), 2)                AS price_vs_market_avg,

        -- Occupancy vs market benchmark
        ROUND(avg_occupied_nights - AVG(avg_occupied_nights) OVER (), 1)
                                                                    AS occupancy_vs_market_avg

    FROM neighborhood_base
),

neighborhood_scored AS (
    SELECT
        *,
        -- Composite opportunity score
        ROUND((price_rank + revenue_rank + occupancy_rank) / 3.0, 1)
                                                                    AS composite_score,

        -- Opportunity tier label
        CASE
            WHEN (price_rank + revenue_rank + occupancy_rank) / 3.0 <= 10
                THEN 'Tier 1 — Premium'
            WHEN (price_rank + revenue_rank + occupancy_rank) / 3.0 <= 25
                THEN 'Tier 2 — Strong'
            WHEN (price_rank + revenue_rank + occupancy_rank) / 3.0 <= 45
                THEN 'Tier 3 — Moderate'
            ELSE
                'Tier 4 — Weak'
        END                                                         AS opportunity_tier

    FROM neighborhood_ranked
)

SELECT
    RANK() OVER (ORDER BY composite_score ASC)                      AS overall_rank,
    neighborhood,
    total_listings,
    opportunity_tier,
    avg_price,
    avg_price_per_bedroom,
    avg_annual_revenue,
    revenue_share_pct,
    avg_occupied_nights,
    occupancy_vs_market_avg,
    price_vs_market_avg,
    avg_rating,
    avg_location_score,
    superhost_rate_pct,
    new_listing_pct,
    composite_score
FROM neighborhood_scored
ORDER BY composite_score ASC
LIMIT 20
"""

# ─── Execute and display ───────────────────────────────────────────────────────
q2_results = pd.read_sql_query(query_2, conn)

print("QUERY 2 — NEIGHBORHOOD PRICE INTELLIGENCE")
print("=" * 65)
print(f"  Neighborhoods analyzed : {len(q2_results)}")
print(f"  Filter: Entire home/apt only | Min 20 listings per neighborhood")
print()
print(q2_results.to_string(index=False))

QUERY 2 — NEIGHBORHOOD PRICE INTELLIGENCE
  Neighborhoods analyzed : 20
  Filter: Entire home/apt only | Min 20 listings per neighborhood

 overall_rank  neighborhood  total_listings opportunity_tier  avg_price  avg_price_per_bedroom  avg_annual_revenue  revenue_share_pct  avg_occupied_nights  occupancy_vs_market_avg  price_vs_market_avg  avg_rating  avg_location_score  superhost_rate_pct  new_listing_pct  composite_score
            1         78732              43 Tier 1 — Premium     392.02                 116.98             25089.0               0.79                 93.7                      8.6               166.67        4.85                4.90                53.5              4.7              5.0
            2         78736              41 Tier 1 — Premium     237.56                 104.19             29600.0               0.89                111.7                     26.6                12.21        4.95                4.91                68.3              4.9              5.7


### Query 2 Results — Neighborhood Price Intelligence

#### Tier 1 — Premium Neighborhoods

| Neighborhood | Avg Price | Annual Revenue | Occupied Nights |
|---|---|---|---|
| 78736 | $237.56 | $29,600 | 111.7 |
| 78732 | $392.02 | $25,089 | 93.7 |
| 78702 | $242.66 | $22,376 | 112.2 |
| 78729 | $188.63 | $19,393 | 108.0 |

#### Key Findings

**High occupancy beats high price.**
`78736` earns the most ($29,600) at only $237/night — 111.7 occupied nights drives the result.

**The overpriced trap.**
`78730` charges $566 but ranks 10th — only 50.8 occupied nights per year.

**Volume capital.**
`78702` — 1,117 listings, 18.3% of all neighborhood revenue, 112.2 occupied nights.

**Underpriced opportunity.**
`78735` leads on occupancy (126.6 nights) but averages only $129 — upside exists.

#### Priority Neighborhoods for a New Host
`78736` — strongest revenue · `78702` — largest market · `78729` — strong occupancy

> **SQL:** Three chained CTEs · `RANK() OVER()` across three dimensions ·
> `HAVING COUNT(*) >= 20` · `CASE WHEN` tier labels · `AVG() OVER()` benchmarks

---
## 2.5 Chapter 2 — Neighborhood Intelligence

### Business Context

Chapter 1 established the market baseline.
Chapter 2 goes deeper into neighborhoods —
answering two questions that directly guide host strategy:

1. Which neighborhoods represent the strongest
   revenue opportunity relative to their competition?
2. Which neighborhoods have demand that consistently
   outpaces supply — signaling pricing power?

---

### Query 3 — Neighborhood Opportunity Matrix

**Business question:**
*Which neighborhoods are underpriced relative to their
occupancy demand — representing the strongest
pricing upside for a new host?*

**DA angle:**
Price and occupancy are inversely correlated in most markets —
higher prices typically suppress demand.
Neighborhoods that break this pattern — high occupancy
despite above-market pricing — signal genuine demand strength.
This query identifies those outliers by plotting each neighborhood's
price premium against its occupancy premium simultaneously.

**BA angle:**
An underpriced high-demand neighborhood is an acquisition opportunity.
A high-priced low-demand neighborhood is a risk.
This matrix separates the two — giving leadership a
data-backed framework for market entry decisions.

In [4]:
# ─── Query 3 — Neighborhood Opportunity Matrix ────────────────────────────────
# Business question: Which neighborhoods are underpriced relative
# to their occupancy demand — representing pricing upside?

query_3 = """
WITH market_averages AS (
    SELECT
        ROUND(AVG(price), 2)                        AS market_avg_price,
        ROUND(AVG(estimated_occupancy_l365d), 1)    AS market_avg_occupancy,
        ROUND(AVG(estimated_revenue_l365d), 0)      AS market_avg_revenue
    FROM listings
    WHERE room_type = 'Entire home/apt'
),

neighborhood_metrics AS (
    SELECT
        neighbourhood_cleansed                                          AS neighborhood,
        COUNT(*)                                                        AS total_listings,
        ROUND(AVG(price), 2)                                            AS avg_price,
        ROUND(AVG(estimated_occupancy_l365d), 1)                        AS avg_occupancy,
        ROUND(AVG(estimated_revenue_l365d), 0)                          AS avg_revenue,
        ROUND(AVG(review_scores_rating), 2)                             AS avg_rating,
        ROUND(AVG(review_scores_location), 2)                           AS avg_location_score,
        ROUND(100.0 * SUM(host_is_superhost) / COUNT(*), 1)             AS superhost_rate,
        ROUND(100.0 * SUM(is_new_listing) / COUNT(*), 1)                AS new_listing_pct,
        ROUND(AVG(amenity_count), 1)                                    AS avg_amenity_count
    FROM listings
    WHERE room_type = 'Entire home/apt'
    GROUP BY neighbourhood_cleansed
    HAVING COUNT(*) >= 20
),

opportunity_matrix AS (
    SELECT
        n.*,
        m.market_avg_price,
        m.market_avg_occupancy,
        m.market_avg_revenue,

        -- Price position vs market
        ROUND(n.avg_price - m.market_avg_price, 2)              AS price_vs_market,
        ROUND(100.0 * (n.avg_price - m.market_avg_price)
            / m.market_avg_price, 1)                            AS price_premium_pct,

        -- Occupancy position vs market
        ROUND(n.avg_occupancy - m.market_avg_occupancy, 1)      AS occupancy_vs_market,
        ROUND(100.0 * (n.avg_occupancy - m.market_avg_occupancy)
            / m.market_avg_occupancy, 1)                        AS occupancy_premium_pct,

        -- Revenue position vs market
        ROUND(n.avg_revenue - m.market_avg_revenue, 0)          AS revenue_vs_market,

        -- Opportunity quadrant classification
        CASE
            WHEN n.avg_price >= m.market_avg_price
             AND n.avg_occupancy >= m.market_avg_occupancy
                THEN 'Quadrant 1 — Premium Demand'
            WHEN n.avg_price < m.market_avg_price
             AND n.avg_occupancy >= m.market_avg_occupancy
                THEN 'Quadrant 2 — Underpriced Opportunity'
            WHEN n.avg_price >= m.market_avg_price
             AND n.avg_occupancy < m.market_avg_occupancy
                THEN 'Quadrant 3 — Overpriced Low Demand'
            ELSE
                'Quadrant 4 — Low Price Low Demand'
        END                                                     AS opportunity_quadrant,

        -- Pricing upside estimate
        -- If underpriced: how much more could be charged
        -- based on market average price
        CASE
            WHEN n.avg_price < m.market_avg_price
                THEN ROUND(m.market_avg_price - n.avg_price, 2)
            ELSE 0
        END                                                     AS pricing_upside_per_night

    FROM neighborhood_metrics n
    CROSS JOIN market_averages m
)

SELECT
    neighborhood,
    total_listings,
    opportunity_quadrant,
    avg_price,
    price_premium_pct                                           AS price_vs_mkt_pct,
    avg_occupancy,
    occupancy_premium_pct                                       AS occupancy_vs_mkt_pct,
    avg_revenue,
    revenue_vs_market,
    pricing_upside_per_night,
    avg_rating,
    avg_location_score,
    superhost_rate,
    new_listing_pct,
    avg_amenity_count
FROM opportunity_matrix
ORDER BY
    opportunity_quadrant ASC,
    avg_revenue DESC
"""

# ─── Execute and display ───────────────────────────────────────────────────────
q3_results = pd.read_sql_query(query_3, conn)

print("QUERY 3 — NEIGHBORHOOD OPPORTUNITY MATRIX")
print("=" * 65)
print(f"  Neighborhoods analyzed : {len(q3_results)}")
print(f"  Market avg price       : ${q3_results['avg_price'].mean():,.2f}")
print(f"  Market avg occupancy   : {q3_results['avg_occupancy'].mean():.1f} nights")
print()

# Print by quadrant
for quadrant in sorted(q3_results['opportunity_quadrant'].unique()):
    subset = q3_results[q3_results['opportunity_quadrant'] == quadrant]
    print(f"\n{quadrant} ({len(subset)} neighborhoods)")
    print("-" * 65)
    print(subset[[
        'neighborhood', 'avg_price', 'price_vs_mkt_pct',
        'avg_occupancy', 'occupancy_vs_mkt_pct',
        'avg_revenue', 'pricing_upside_per_night'
    ]].to_string(index=False))

QUERY 3 — NEIGHBORHOOD OPPORTUNITY MATRIX
  Neighborhoods analyzed : 39
  Market avg price       : $225.35
  Market avg occupancy   : 85.1 nights


Quadrant 1 — Premium Demand (4 neighborhoods)
-----------------------------------------------------------------
 neighborhood  avg_price  price_vs_mkt_pct  avg_occupancy  occupancy_vs_mkt_pct  avg_revenue  pricing_upside_per_night
        78736     237.56               7.0          111.7                  27.2      29600.0                       0.0
        78732     392.02              76.6           93.7                   6.7      25089.0                       0.0
        78702     242.66               9.3          112.2                  27.8      22376.0                       0.0
        78704     256.64              15.6           88.9                   1.3      17110.0                       0.0

Quadrant 2 — Underpriced Opportunity (13 neighborhoods)
-----------------------------------------------------------------
 neighborhood  avg_pri

### Query 3 Results — Neighborhood Opportunity Matrix

**Market baseline:** Avg price $225.35 · Avg occupancy 85.1 nights

---

#### Quadrant 1 — Premium Demand (4 neighborhoods)
High price + high occupancy = best market conditions

| Neighborhood | Avg Price | Occupancy | Annual Revenue |
|---|---|---|---|
| 78702 | $242.66 | 112.2 nights | $22,376 |
| 78704 | $256.64 | 88.9 nights | $17,110 |
| 78732 | $392.02 | 93.7 nights | $25,089 |
| 78736 | $237.56 | 111.7 nights | $29,600 |

---

#### Quadrant 2 — Underpriced Opportunity (13 neighborhoods)
Below-market price + above-market occupancy = pricing upside exists

Top opportunities by pricing upside per night:

| Neighborhood | Avg Price | Occupancy | Upside/Night | Annual Revenue |
|---|---|---|---|---|
| 78751 | $126.11 | 114.6 nights | +$95.86 | $12,624 |
| 78735 | $129.40 | 126.6 nights | +$92.57 | $15,020 |
| 78705 | $137.68 | 93.2 nights | +$84.29 | $11,007 |
| 78729 | $188.63 | 108.0 nights | +$33.34 | $19,393 |

**78735 has the highest occupancy in the entire dataset (126.6 nights)
at only $129/night — the strongest underpriced signal in Austin.**

---

#### Quadrant 3 — Overpriced Low Demand (8 neighborhoods)
High price + low occupancy = avoid

| Neighborhood | Avg Price | Occupancy | Annual Revenue |
|---|---|---|---|
| 78730 | $566.26 | 50.8 nights | $22,424 |
| 78746 | $546.87 | 51.2 nights | $20,849 |
| 78733 | $429.60 | 60.9 nights | $17,226 |

Premium pricing without demand produces mediocre returns.

---

#### Quadrant 4 — Low Price Low Demand (14 neighborhoods)
Below-market price + below-market occupancy = weakest segment

---

#### Strategic Implication

**Q2 neighborhoods are the priority for a new host.**
Strong demand already exists — pricing at or above
the neighborhood average captures occupancy
while improving revenue per night.

> **SQL:** `CROSS JOIN` market averages into neighborhood metrics ·
> `CASE WHEN` across two dimensions builds four quadrant labels ·
> Pricing upside calculated as market average minus neighborhood average

---
### Query 4 — Property Type Deep Dive

**Business question:**
*Beyond room type — which specific property types
command the strongest price premiums and revenue,
and does property type explain price variation
that room type alone cannot?*

**DA angle:**
Room type is a broad category — "Entire home/apt" contains
everything from a studio apartment to a 10-bedroom mansion.
Property type adds a second layer of segmentation —
distinguishing between entire condos, houses, guesthouses,
townhouses, and cabins within the same room type.
This query tests whether property type is a statistically
meaningful pricing variable beyond room type alone.

**BA angle:**
A property management company acquiring Austin properties
needs to know whether property type influences
revenue independently of room type.
If a guesthouse and a condo in the same neighborhood
charge the same price — property type is irrelevant.
If they diverge significantly — acquisition strategy
must account for property type explicitly.

In [5]:
# ─── Query 4 — Property Type Deep Dive ───────────────────────────────────────
# Business question: Which property types command the strongest
# price premiums beyond what room type alone explains?

query_4 = """
WITH market_baseline AS (
    SELECT
        ROUND(AVG(price), 2)                                    AS market_avg_price,
        ROUND(AVG(estimated_revenue_l365d), 0)                  AS market_avg_revenue,
        ROUND(AVG(estimated_occupancy_l365d), 1)                AS market_avg_occupancy
    FROM listings
    WHERE room_type = 'Entire home/apt'
),

property_metrics AS (
    SELECT
        property_type,

        -- Volume
        COUNT(*)                                                AS total_listings,

        -- Price metrics
        ROUND(AVG(price), 2)                                    AS avg_price,
        ROUND(AVG(price_per_bedroom), 2)                        AS avg_price_per_bedroom,
        ROUND(MIN(price), 2)                                    AS min_price,
        ROUND(MAX(price), 2)                                    AS max_price,

        -- Revenue metrics
        ROUND(AVG(estimated_revenue_l365d), 0)                  AS avg_annual_revenue,

        -- Occupancy
        ROUND(AVG(estimated_occupancy_l365d), 1)                AS avg_occupancy,

        -- Guest capacity
        ROUND(AVG(accommodates), 1)                             AS avg_accommodates,
        ROUND(AVG(bedrooms), 1)                                 AS avg_bedrooms,

        -- Quality
        ROUND(AVG(review_scores_rating), 2)                     AS avg_rating,
        ROUND(100.0 * SUM(host_is_superhost) / COUNT(*), 1)     AS superhost_rate,

        -- Amenities
        ROUND(AVG(amenity_count), 1)                            AS avg_amenity_count

    FROM listings
    WHERE room_type = 'Entire home/apt'
    GROUP BY property_type
    HAVING COUNT(*) >= 15
),

property_ranked AS (
    SELECT
        p.*,
        b.market_avg_price,
        b.market_avg_revenue,
        b.market_avg_occupancy,

        -- Price premium vs market
        ROUND(p.avg_price - b.market_avg_price, 2)              AS price_vs_market,
        ROUND(100.0 * (p.avg_price - b.market_avg_price)
            / b.market_avg_price, 1)                            AS price_premium_pct,

        -- Revenue premium vs market
        ROUND(p.avg_annual_revenue - b.market_avg_revenue, 0)   AS revenue_vs_market,

        -- Ranks
        RANK() OVER (ORDER BY p.avg_price DESC)                 AS price_rank,
        RANK() OVER (ORDER BY p.avg_annual_revenue DESC)        AS revenue_rank,
        RANK() OVER (ORDER BY p.avg_occupancy DESC)             AS occupancy_rank

    FROM property_metrics p
    CROSS JOIN market_baseline b
)

SELECT
    price_rank                                                  AS rank,
    property_type,
    total_listings,
    avg_price,
    price_premium_pct                                           AS price_vs_mkt_pct,
    avg_price_per_bedroom,
    avg_annual_revenue,
    revenue_vs_market,
    avg_occupancy,
    avg_accommodates,
    avg_bedrooms,
    avg_rating,
    superhost_rate,
    avg_amenity_count
FROM property_ranked
ORDER BY price_rank ASC
"""

# ─── Execute and display ───────────────────────────────────────────────────────
q4_results = pd.read_sql_query(query_4, conn)

print("QUERY 4 — PROPERTY TYPE DEEP DIVE")
print("=" * 65)
print(f"  Property types analyzed : {len(q4_results)}")
print(f"  Filter: Entire home/apt · Min 15 listings per property type")
print()
print(q4_results.to_string(index=False))

QUERY 4 — PROPERTY TYPE DEEP DIVE
  Property types analyzed : 18
  Filter: Entire home/apt · Min 15 listings per property type

 rank             property_type  total_listings  avg_price  price_vs_mkt_pct  avg_price_per_bedroom  avg_annual_revenue  revenue_vs_market  avg_occupancy  avg_accommodates  avg_bedrooms  avg_rating  superhost_rate  avg_amenity_count
    1              Entire villa              30     555.43             150.2                 153.53             30991.0            15787.0           54.6              10.9           4.2        4.94            76.7               61.0
    2        Room in aparthotel              16     311.88              40.5                 214.97             21398.0             6194.0           85.5               3.4           1.2        4.49             6.3               42.5
    3               Entire home            4440     294.45              32.7                  91.96             20097.0             4893.0           87.7               7.6  

### Query 4 Results — Property Type Deep Dive

**18 property types analyzed · Entire home/apt segment only**

#### Standout Findings

**Villas charge the most — but earn it differently.**
Entire villa averages $555/night — 150% above market.
At 54.6 occupied nights, annual revenue ($30,991) leads all types.
Large capacity (10.9 guests avg) justifies the premium.

**Cabins and guesthouses — the occupancy champions.**
Tiny home: 132.7 occupied nights · Guest suite: 122.9 · 
Guesthouse: 118.5 · Cabin: 103.1.
All priced below market — all generating strong revenue
through volume rather than premium pricing.

**Entire home dominates by volume.**
4,440 listings — the backbone of the Austin market.
$294 avg price, 87.7 occupied nights, $20,097 annual revenue.
Above market on price AND occupancy — the strongest balanced performer.

**Serviced apartments and resorts underperform.**
Entire resort: 0 occupied nights recorded — data anomaly.
Serviced apartment: only 39.7 occupied nights despite moderate pricing.
These property types show the weakest demand signals.

**Price per bedroom reveals the real premium.**
Room in aparthotel: $214/bedroom · Serviced apartment: $171/bedroom ·
Entire villa: $153/bedroom.
Smaller units command higher per-bedroom rates —
guests pay a convenience premium for compact spaces.

#### Strategic Implication

Entire home and entire cabin represent the strongest
balanced opportunity — competitive pricing, proven demand,
and consistent occupancy above market average.

> **SQL:** `CROSS JOIN` market baseline · `RANK() OVER()` across
> three dimensions · `HAVING COUNT(*) >= 15` for reliability

---
## 2.6 Chapter 3 — Price Drivers

### Business Context

Chapters 1 and 2 established where to list.
Chapter 3 answers what drives price within a listing —
the variables a host can actually control.

Two questions drive this chapter:

1. Does superhost status command a measurable price premium —
   and is that premium consistent across room types?
2. Which specific amenities produce the highest
   price premium per dollar of investment?

These findings translate directly into host action items.

---

### Query 5 — Superhost Premium Analysis

**Business question:**
*Do superhosts charge more, earn more, and occupy more nights
than non-superhosts — and does this premium hold consistently
across all room types and neighborhood tiers?*

**DA angle:**
Superhost status is a platform-assigned quality signal.
The analytical question is whether the market rewards that signal
with measurable price and revenue premiums —
or whether superhost status is merely correlated with
other variables like experience and amenity count.
This query isolates the superhost effect across multiple dimensions.

**BA angle:**
If superhost status commands a consistent revenue premium,
it becomes a measurable KPI for a property management company —
and a concrete target for new host onboarding strategy.
The premium quantified here becomes a business case
for investing in the behaviors that earn superhost status.

In [6]:
# ─── Query 5 — Superhost Premium Analysis ────────────────────────────────────
# Business question: Do superhosts charge more, earn more,
# and occupy more nights — consistently across room types?

query_5 = """
WITH superhost_overall AS (
    SELECT
        CASE WHEN host_is_superhost = 1
            THEN 'Superhost'
            ELSE 'Non-Superhost'
        END                                                         AS host_type,

        COUNT(*)                                                    AS total_listings,

        -- Price metrics
        ROUND(AVG(price), 2)                                        AS avg_price,
        ROUND(AVG(price_per_bedroom), 2)                            AS avg_price_per_bedroom,

        -- Revenue metrics
        ROUND(AVG(estimated_revenue_l365d), 0)                      AS avg_annual_revenue,

        -- Occupancy
        ROUND(AVG(estimated_occupancy_l365d), 1)                    AS avg_occupancy,

        -- Quality signals
        ROUND(AVG(review_scores_rating), 2)                         AS avg_rating,
        ROUND(AVG(review_scores_cleanliness), 2)                    AS avg_cleanliness,
        ROUND(AVG(review_scores_location), 2)                       AS avg_location,
        ROUND(AVG(reviews_per_month), 2)                            AS avg_reviews_per_month,

        -- Host profile
        ROUND(AVG(host_experience_years), 1)                        AS avg_experience_years,
        ROUND(AVG(amenity_count), 1)                                AS avg_amenity_count,
        ROUND(AVG(calculated_host_listings_count), 1)               AS avg_listings_count

    FROM listings
    GROUP BY host_is_superhost
),

superhost_by_roomtype AS (
    SELECT
        room_type,
        CASE WHEN host_is_superhost = 1
            THEN 'Superhost'
            ELSE 'Non-Superhost'
        END                                                         AS host_type,
        COUNT(*)                                                    AS total_listings,
        ROUND(AVG(price), 2)                                        AS avg_price,
        ROUND(AVG(estimated_revenue_l365d), 0)                      AS avg_annual_revenue,
        ROUND(AVG(estimated_occupancy_l365d), 1)                    AS avg_occupancy,
        ROUND(AVG(review_scores_rating), 2)                         AS avg_rating
    FROM listings
    GROUP BY room_type, host_is_superhost
),

premium_calc AS (
    SELECT
        s.host_type,
        s.total_listings,
        s.avg_price,
        s.avg_price_per_bedroom,
        s.avg_annual_revenue,
        s.avg_occupancy,
        s.avg_rating,
        s.avg_cleanliness,
        s.avg_location,
        s.avg_reviews_per_month,
        s.avg_experience_years,
        s.avg_amenity_count,
        s.avg_listings_count,

        -- Price premium calculation
        ROUND(s.avg_price - AVG(s.avg_price) OVER (), 2)            AS price_vs_overall_avg,
        ROUND(100.0 * (s.avg_price - AVG(s.avg_price) OVER ())
            / AVG(s.avg_price) OVER (), 1)                          AS price_premium_pct,

        -- Revenue premium
        ROUND(s.avg_annual_revenue -
            AVG(s.avg_annual_revenue) OVER (), 0)                   AS revenue_vs_avg,

        -- Occupancy premium
        ROUND(s.avg_occupancy -
            AVG(s.avg_occupancy) OVER (), 1)                        AS occupancy_vs_avg

    FROM superhost_overall s
)

SELECT * FROM premium_calc
ORDER BY avg_price DESC;

-- Superhost breakdown by room type
SELECT
    room_type,
    host_type,
    total_listings,
    avg_price,
    avg_annual_revenue,
    avg_occupancy,
    avg_rating
FROM superhost_by_roomtype
ORDER BY room_type, host_type DESC
"""

# ─── Execute both parts ───────────────────────────────────────────────────────
# Part 1 — Overall superhost premium
query_5a = query_5.split(';')[0]
q5_overall = pd.read_sql_query(query_5a, conn)

# Part 2 — By room type
query_5b = """
    SELECT
        room_type,
        CASE WHEN host_is_superhost = 1
            THEN 'Superhost'
            ELSE 'Non-Superhost'
        END                                                     AS host_type,
        COUNT(*)                                                AS total_listings,
        ROUND(AVG(price), 2)                                    AS avg_price,
        ROUND(AVG(estimated_revenue_l365d), 0)                  AS avg_annual_revenue,
        ROUND(AVG(estimated_occupancy_l365d), 1)                AS avg_occupancy,
        ROUND(AVG(review_scores_rating), 2)                     AS avg_rating
    FROM listings
    GROUP BY room_type, host_is_superhost
    ORDER BY room_type, host_is_superhost DESC
"""
q5_roomtype = pd.read_sql_query(query_5b, conn)

print("QUERY 5 — SUPERHOST PREMIUM ANALYSIS")
print("=" * 65)
print("\nPART A — Overall Superhost vs Non-Superhost")
print("-" * 65)
print(q5_overall.to_string(index=False))

print("\nPART B — Superhost Premium by Room Type")
print("-" * 65)
print(q5_roomtype.to_string(index=False))

QUERY 5 — SUPERHOST PREMIUM ANALYSIS

PART A — Overall Superhost vs Non-Superhost
-----------------------------------------------------------------
    host_type  total_listings  avg_price  avg_price_per_bedroom  avg_annual_revenue  avg_occupancy  avg_rating  avg_cleanliness  avg_location  avg_reviews_per_month  avg_experience_years  avg_amenity_count  avg_listings_count  price_vs_overall_avg  price_premium_pct  revenue_vs_avg  occupancy_vs_avg
    Superhost            5468     205.27                  92.29             19627.0          115.0        4.90             4.88          4.86                   2.01                   8.9               49.3                 8.9                  0.18                0.1          6383.0              33.8
Non-Superhost            4934     204.91                 107.65              6862.0           47.5        4.81             4.79          4.81                   0.91                   8.0               38.3                10.8                 -0.18   

### Query 5 Results — Superhost Premium Analysis

#### The Counter-Intuitive Finding

**Superhosts do not charge more per night — they earn more per year.**

| Metric | Superhost | Non-Superhost | Difference |
|---|---|---|---|
| Avg nightly price | $205.27 | $204.91 | +$0.36 |
| Avg annual revenue | $19,627 | $6,862 | +$12,765 |
| Avg occupied nights | 115.0 | 47.5 | +67.5 nights |
| Avg rating | 4.90 | 4.81 | +0.09 |
| Avg amenity count | 49.3 | 38.3 | +11 amenities |

**The price difference between superhosts and non-superhosts
is virtually zero ($0.36). The revenue difference is $12,765.**
The entire gap is explained by occupancy — not price.

---

#### Entire Home Segment — The Sharpest Contrast

| Host Type | Avg Price | Occupied Nights | Annual Revenue |
|---|---|---|---|
| Superhost | $218.52 | 117.5 | $21,007 |
| Non-Superhost | $226.25 | 51.0 | $8,004 |

Non-superhosts charge $7.73 more per night
yet earn $13,003 less per year.
Overpricing without the quality signal to support it
produces strong nightly rates and empty calendars.

---

#### Key Findings

**Superhost status is an occupancy signal — not a price signal.**
The market does not reward superhost status with higher prices.
It rewards it with more bookings.
67.5 additional occupied nights per year at similar pricing
produces 186% more annual revenue.

**Amenities separate the segments.**
Superhosts average 49.3 amenities vs 38.3 for non-superhosts.
An 11-amenity gap suggests superhosts invest more
in listing quality — which converts to higher occupancy.

**Experience matters modestly.**
Superhosts average 8.9 years experience vs 8.0 for non-superhosts.
The gap is small — experience alone does not create superhost performance.
Listing quality and guest experience drive the difference.

#### Strategic Implication

A new host should not price above neighborhood averages
expecting superhost premium to follow.
The data shows the opposite sequence —
price competitively → earn reviews → build occupancy →
achieve superhost status → sustain revenue through bookings.

> **SQL:** `CASE WHEN host_is_superhost` for binary segmentation ·
> `AVG() OVER()` for overall benchmark comparison ·
> Two-part query — overall premium then room type breakdown

---
### Query 6 — Amenity Price Premium Matrix

**Business question:**
*Which specific amenities command the highest price premium —
and which represent the strongest return on investment
for a new host looking to maximize nightly rate?*

**DA angle:**
Amenity features were engineered as binary columns in Notebook 1.
This query uses those columns to calculate the exact price premium
each amenity commands — controlling for listing count
to ensure statistical reliability.
The result is a ranked amenity ROI matrix —
sortable by price premium, revenue premium, or occupancy impact.

**BA angle:**
Every amenity represents a host investment decision.
A pool costs significantly more to install than a BBQ grill.
If both command similar price premiums, the BBQ grill
delivers superior ROI.
This query gives a property management company
the data to prioritize capital improvements
across an Austin portfolio.

In [7]:
# ─── Query 6 — Amenity Price Premium Matrix ───────────────────────────────────
# Business question: Which amenities command the highest price
# premium and represent the strongest host investment ROI?

query_6 = """
WITH market_baseline AS (
    SELECT
        ROUND(AVG(price), 2)                                    AS market_avg_price,
        ROUND(AVG(estimated_revenue_l365d), 0)                  AS market_avg_revenue,
        ROUND(AVG(estimated_occupancy_l365d), 1)                AS market_avg_occupancy,
        COUNT(*)                                                AS total_market_listings
    FROM listings
    WHERE room_type = 'Entire home/apt'
),

amenity_has_wifi AS (
    SELECT 'WiFi' AS amenity, COUNT(*) AS listings_with,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(AVG(estimated_revenue_l365d), 0) AS avg_revenue,
        ROUND(AVG(estimated_occupancy_l365d), 1) AS avg_occupancy,
        ROUND(AVG(review_scores_rating), 2) AS avg_rating
    FROM listings WHERE has_wifi = 1 AND room_type = 'Entire home/apt'
UNION ALL
    SELECT 'No WiFi', COUNT(*),
        ROUND(AVG(price), 2), ROUND(AVG(estimated_revenue_l365d), 0),
        ROUND(AVG(estimated_occupancy_l365d), 1), ROUND(AVG(review_scores_rating), 2)
    FROM listings WHERE has_wifi = 0 AND room_type = 'Entire home/apt'
),

amenity_stats AS (
    SELECT 'Pool' AS amenity,
        ROUND(AVG(CASE WHEN has_pool = 1 THEN price END), 2)            AS with_amenity_price,
        ROUND(AVG(CASE WHEN has_pool = 0 THEN price END), 2)            AS without_amenity_price,
        ROUND(AVG(CASE WHEN has_pool = 1 THEN estimated_revenue_l365d END), 0)
                                                                        AS with_amenity_revenue,
        ROUND(AVG(CASE WHEN has_pool = 0 THEN estimated_revenue_l365d END), 0)
                                                                        AS without_amenity_revenue,
        ROUND(AVG(CASE WHEN has_pool = 1 THEN estimated_occupancy_l365d END), 1)
                                                                        AS with_amenity_occupancy,
        ROUND(AVG(CASE WHEN has_pool = 0 THEN estimated_occupancy_l365d END), 1)
                                                                        AS without_amenity_occupancy,
        SUM(CASE WHEN has_pool = 1 THEN 1 ELSE 0 END)                   AS listings_with
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'Hot Tub',
        ROUND(AVG(CASE WHEN has_hot_tub = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_hot_tub = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_hot_tub = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_hot_tub = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_hot_tub = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_hot_tub = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_hot_tub = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'EV Charger',
        ROUND(AVG(CASE WHEN has_ev_charger = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_ev_charger = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_ev_charger = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_ev_charger = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_ev_charger = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_ev_charger = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_ev_charger = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'Gym',
        ROUND(AVG(CASE WHEN has_gym = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_gym = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_gym = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_gym = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_gym = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_gym = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_gym = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'BBQ Grill',
        ROUND(AVG(CASE WHEN has_bbq = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_bbq = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_bbq = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_bbq = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_bbq = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_bbq = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_bbq = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'Fire Pit',
        ROUND(AVG(CASE WHEN has_fire_pit = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_fire_pit = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_fire_pit = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_fire_pit = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_fire_pit = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_fire_pit = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_fire_pit = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'Free Parking',
        ROUND(AVG(CASE WHEN has_free_parking = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_free_parking = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_free_parking = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_free_parking = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_free_parking = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_free_parking = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_free_parking = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'Private Entrance',
        ROUND(AVG(CASE WHEN has_private_entrance = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_private_entrance = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_private_entrance = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_private_entrance = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_private_entrance = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_private_entrance = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_private_entrance = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'Pets Allowed',
        ROUND(AVG(CASE WHEN has_pets_allowed = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_pets_allowed = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_pets_allowed = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_pets_allowed = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_pets_allowed = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_pets_allowed = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_pets_allowed = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'

    UNION ALL SELECT 'Washer',
        ROUND(AVG(CASE WHEN has_washer = 1 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_washer = 0 THEN price END), 2),
        ROUND(AVG(CASE WHEN has_washer = 1 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_washer = 0 THEN estimated_revenue_l365d END), 0),
        ROUND(AVG(CASE WHEN has_washer = 1 THEN estimated_occupancy_l365d END), 1),
        ROUND(AVG(CASE WHEN has_washer = 0 THEN estimated_occupancy_l365d END), 1),
        SUM(CASE WHEN has_washer = 1 THEN 1 ELSE 0 END)
    FROM listings WHERE room_type = 'Entire home/apt'
),

amenity_premiums AS (
    SELECT
        amenity,
        listings_with,
        with_amenity_price,
        without_amenity_price,
        ROUND(with_amenity_price - without_amenity_price, 2)    AS price_premium,
        ROUND(100.0 * (with_amenity_price - without_amenity_price)
            / without_amenity_price, 1)                         AS price_premium_pct,
        with_amenity_revenue,
        without_amenity_revenue,
        ROUND(with_amenity_revenue - without_amenity_revenue, 0) AS revenue_premium,
        with_amenity_occupancy,
        without_amenity_occupancy,
        ROUND(with_amenity_occupancy - without_amenity_occupancy, 1)
                                                                AS occupancy_premium,
        RANK() OVER (ORDER BY
            (with_amenity_price - without_amenity_price) DESC)  AS price_premium_rank
    FROM amenity_stats
)

SELECT
    price_premium_rank                                          AS rank,
    amenity,
    listings_with,
    with_amenity_price,
    without_amenity_price,
    price_premium,
    price_premium_pct,
    with_amenity_revenue,
    revenue_premium,
    with_amenity_occupancy,
    occupancy_premium
FROM amenity_premiums
ORDER BY price_premium_rank
"""

# ─── Execute and display ───────────────────────────────────────────────────────
q6_results = pd.read_sql_query(query_6, conn)

print("QUERY 6 — AMENITY PRICE PREMIUM MATRIX")
print("=" * 65)
print(f"  Amenities analyzed : {len(q6_results)}")
print(f"  Segment            : Entire home/apt only")
print()
print(q6_results.to_string(index=False))

QUERY 6 — AMENITY PRICE PREMIUM MATRIX
  Amenities analyzed : 10
  Segment            : Entire home/apt only

 rank          amenity  listings_with  with_amenity_price  without_amenity_price  price_premium  price_premium_pct  with_amenity_revenue  revenue_premium  with_amenity_occupancy  occupancy_premium
    1          Hot Tub            870              430.86                 199.65         231.21              115.8               30773.0          17232.0                    87.6               -0.2
    2        BBQ Grill           4002              286.04                 170.80         115.24               67.5               17954.0           4946.0                    78.6              -16.6
    3           Washer           8027              233.51                 128.00         105.51               82.4               15931.0           6647.0                    86.9               -8.7
    4             Pool           3387              283.89                 184.69          99.20       

### Query 6 Results — Amenity Price Premium Matrix

**Segment: Entire home/apt only · 10 amenities analyzed**

| Rank | Amenity | Price Premium | Revenue Premium | Occupancy Impact |
|---|---|---|---|---|
| 1 | Hot Tub | +$231.21 (+115.8%) | +$17,232 | -0.2 nights |
| 2 | BBQ Grill | +$115.24 (+67.5%) | +$4,946 | -16.6 nights |
| 3 | Washer | +$105.51 (+82.4%) | +$6,647 | -8.7 nights |
| 4 | Pool | +$99.20 (+53.7%) | +$5,575 | -15.7 nights |
| 5 | Fire Pit | +$86.48 (+43.2%) | +$4,768 | -10.1 nights |
| 6 | EV Charger | +$29.82 (+13.6%) | +$8,440 | +22.7 nights |
| 7 | Free Parking | +$13.38 (+6.3%) | -$1,057 | -2.7 nights |
| 8 | Private Entrance | -$12.38 (-5.4%) | +$5,347 | +31.6 nights |
| 9 | Pets Allowed | -$13.58 (-6.0%) | +$2,449 | +10.8 nights |
| 10 | Gym | -$16.39 (-7.3%) | -$3,428 | -20.5 nights |

---

#### Key Findings

**Hot tub is the single most powerful pricing amenity in Austin.**
A $231/night premium over listings without one — 115.8% above baseline.
At $30,773 average annual revenue, hot tub listings earn
more than double the market average.
Only 870 entire home listings have one — scarcity amplifies the premium.

**EV charger is the highest ROI emerging amenity.**
The smallest price premium ($29.82) but the strongest
occupancy impact (+22.7 nights) and third highest revenue premium ($8,440).
Only 739 listings offer one — first-mover advantage exists
before saturation reduces the signal.

**The counterintuitive amenities — private entrance, pets, gym.**
All three are associated with LOWER nightly prices.
Private entrance (-$12.38) and pets allowed (-$13.58) attract
guests who prioritize flexibility over premium features —
a different demand segment entirely.
Gym access (-$16.39, -20.5 occupancy nights) shows
the weakest performance of all amenities analyzed.

**Free parking is table stakes — not a differentiator.**
At 74.7% market penetration, free parking is expected.
The $13.38 premium it commands is statistically negligible.
A new host should not invest in parking infrastructure
expecting a meaningful pricing return.

**BBQ grill is Austin's highest ROI outdoor investment.**
$115 price premium at 42.6% market penetration —
still uncommon enough to differentiate.
Fire pit ($86 premium, 24.6% penetration) offers
similar outdoor appeal at likely lower investment cost.

#### Strategic Implication — Host Investment Priority

| Priority | Amenity | Rationale |
|---|---|---|
| High | Hot tub | Highest price and revenue premium |
| High | EV charger | Best occupancy ROI, low competition |
| Medium | BBQ grill | Strong price premium, Austin culture fit |
| Medium | Fire pit | Good premium, lower cost than pool |
| Low | Pool | Strong premium but high installation cost |
| Avoid | Gym | Negative price and revenue impact |

> **SQL:** `CASE WHEN` inside `AVG()` isolates with/without
> amenity groups in a single pass · `UNION ALL` stacks
> 10 amenity calculations into one result set ·
> `RANK() OVER()` orders by price premium descending

---
## 2.7 Chapter 4 — Host Intelligence

### Business Context

Chapters 1 through 3 answered what to list and where.
Chapter 4 answers who the competition is —
and what the top performers do differently.

Two questions drive this chapter:

1. What separates the top 10% of revenue earners
   from the bottom 10% — and can those patterns
   be replicated by a new host?
2. Does host experience translate into measurable
   price and revenue improvement over time —
   or does performance plateau early?

---

### Query 7 — Top Performer Profiling

**Business question:**
*What do the top 10% revenue hosts look like
compared to the bottom 10% — across price,
occupancy, amenities, experience, and superhost status?*

**DA angle:**
Using `NTILE(10)` window function, listings are divided
into 10 equal revenue deciles. Decile 10 represents
the top 10% of earners. Decile 1 represents the bottom 10%.
Comparing these two extremes surfaces the characteristics
that most strongly separate high performers from low performers —
without the noise of the middle of the distribution.

**BA angle:**
Top performer profiling is the foundation of
any host onboarding strategy. If the top 10% consistently
share specific characteristics — superhost status,
amenity count, neighborhood, property type —
those characteristics become the target profile
for new host acquisition and development.

In [8]:
# ─── Query 7 — Top Performer Profiling ───────────────────────────────────────
# Business question: What separates the top 10% revenue hosts
# from the bottom 10% across all key dimensions?

query_7 = """
WITH revenue_deciles AS (
    SELECT
        *,
        NTILE(10) OVER (ORDER BY estimated_revenue_l365d ASC)   AS revenue_decile
    FROM listings
    WHERE room_type = 'Entire home/apt'
    AND estimated_revenue_l365d > 0
),

decile_profiles AS (
    SELECT
        CASE
            WHEN revenue_decile = 10 THEN 'Top 10% — High Performers'
            WHEN revenue_decile = 1  THEN 'Bottom 10% — Low Performers'
        END                                                     AS performer_tier,

        COUNT(*)                                                AS total_listings,

        -- Price metrics
        ROUND(AVG(price), 2)                                    AS avg_price,
        ROUND(AVG(price_per_bedroom), 2)                        AS avg_price_per_bedroom,

        -- Revenue metrics
        ROUND(AVG(estimated_revenue_l365d), 0)                  AS avg_annual_revenue,
        ROUND(MIN(estimated_revenue_l365d), 0)                  AS min_annual_revenue,
        ROUND(MAX(estimated_revenue_l365d), 0)                  AS max_annual_revenue,

        -- Occupancy
        ROUND(AVG(estimated_occupancy_l365d), 1)                AS avg_occupancy,

        -- Property features
        ROUND(AVG(accommodates), 1)                             AS avg_accommodates,
        ROUND(AVG(bedrooms), 1)                                 AS avg_bedrooms,
        ROUND(AVG(bathrooms), 1)                                AS avg_bathrooms,
        ROUND(AVG(amenity_count), 1)                            AS avg_amenity_count,

        -- Amenity flags
        ROUND(100.0 * SUM(has_pool) / COUNT(*), 1)              AS pct_has_pool,
        ROUND(100.0 * SUM(has_hot_tub) / COUNT(*), 1)           AS pct_has_hot_tub,
        ROUND(100.0 * SUM(has_bbq) / COUNT(*), 1)               AS pct_has_bbq,
        ROUND(100.0 * SUM(has_free_parking) / COUNT(*), 1)      AS pct_free_parking,
        ROUND(100.0 * SUM(has_ev_charger) / COUNT(*), 1)        AS pct_ev_charger,

        -- Host profile
        ROUND(AVG(host_experience_years), 1)                    AS avg_experience_years,
        ROUND(100.0 * SUM(host_is_superhost) / COUNT(*), 1)     AS superhost_rate,
        ROUND(AVG(calculated_host_listings_count), 1)           AS avg_listings_count,

        -- Quality metrics
        ROUND(AVG(review_scores_rating), 2)                     AS avg_rating,
        ROUND(AVG(review_scores_cleanliness), 2)                AS avg_cleanliness,
        ROUND(AVG(reviews_per_month), 2)                        AS avg_reviews_per_month,

        -- New listing flag
        ROUND(100.0 * SUM(is_new_listing) / COUNT(*), 1)        AS pct_new_listing

    FROM revenue_deciles
    WHERE revenue_decile IN (1, 10)
    GROUP BY revenue_decile
)

SELECT * FROM decile_profiles
WHERE performer_tier IS NOT NULL
ORDER BY avg_annual_revenue DESC
"""

# ─── Execute and display ───────────────────────────────────────────────────────
q7_results = pd.read_sql_query(query_7, conn)

print("QUERY 7 — TOP PERFORMER PROFILING")
print("=" * 65)
print(f"  Segment : Entire home/apt · Revenue > 0")
print(f"  Method  : NTILE(10) revenue deciles — comparing decile 10 vs decile 1")
print()

# Display transposed for easier reading
print(q7_results.T.to_string(header=False))

QUERY 7 — TOP PERFORMER PROFILING
  Segment : Entire home/apt · Revenue > 0
  Method  : NTILE(10) revenue deciles — comparing decile 10 vs decile 1

performer_tier         Top 10% — High Performers  Bottom 10% — Low Performers
total_listings                               705                          706
avg_price                                 470.32                       130.43
avg_price_per_bedroom                     129.48                        84.58
avg_annual_revenue                       71525.0                       1243.0
min_annual_revenue                       40545.0                        144.0
max_annual_revenue                      407745.0                       2220.0
avg_occupancy                              184.6                         11.9
avg_accommodates                            10.2                          4.7
avg_bedrooms                                 3.9                          1.7
avg_bathrooms                                2.9                       

### Query 7 Results — Top Performer Profiling

| Metric | Top 10% | Bottom 10% |
|---|---|---|
| Avg annual revenue | $71,525 | $1,243 |
| Avg occupied nights | 184.6 | 11.9 |
| Avg nightly price | $470.32 | $130.43 |
| Superhost rate | 82.1% | 32.7% |
| Avg bedrooms | 3.9 | 1.7 |
| Avg amenity count | 58.8 | 39.6 |
| Pool | 62.6% | 37.0% |
| Hot tub | 33.6% | 7.1% |
| Avg rating | 4.92 | 4.71 |

**Revenue gap: 57x. Occupancy gap: 15x. Price gap: 3.6x.**

Top performers run larger properties (10.2 guest capacity vs 4.7),
invest heavily in premium amenities, and maintain 82.1% superhost rate.
Bottom performers average MORE listings (10.1 vs 8.8) —
volume without quality consistently underperforms.

**Target profile for a new host:**
4+ bedrooms · hot tub · pool · BBQ · price competitively
to build occupancy → reviews → superhost status.

> **SQL:** `NTILE(10) OVER()` splits listings into revenue deciles ·
> amenity penetration via `SUM(has_pool) / COUNT(*)`

---
### Query 8 — Host Experience Curve

**Business question:**
*Does host experience translate into measurable price
and revenue improvement over time — or does performance
plateau after the first few years?*

**DA angle:**
Host experience in years is a continuous variable
engineered from `host_since` in Notebook 1.
This query buckets experience into meaningful cohorts
and tracks price, revenue, occupancy, and superhost rate
across each cohort — revealing whether the experience curve
is linear, logarithmic, or flat after a certain threshold.

**BA angle:**
If revenue grows consistently with experience,
host retention becomes a core business strategy.
If performance plateaus after year 3,
the acquisition of experienced hosts beyond that threshold
delivers diminishing returns — and onboarding new hosts
to year 3 quickly becomes the priority investment.

In [9]:
# ─── Query 8 — Host Experience Curve ─────────────────────────────────────────
# Business question: Does host experience translate into measurable
# price and revenue improvement over time?

query_8 = """
WITH experience_cohorts AS (
    SELECT
        *,
        CASE
            WHEN host_experience_years < 1  THEN '1. Under 1 year'
            WHEN host_experience_years < 2  THEN '2. 1-2 years'
            WHEN host_experience_years < 3  THEN '3. 2-3 years'
            WHEN host_experience_years < 5  THEN '4. 3-5 years'
            WHEN host_experience_years < 8  THEN '5. 5-8 years'
            WHEN host_experience_years < 12 THEN '6. 8-12 years'
            ELSE                                 '7. 12+ years'
        END                                                     AS experience_cohort
    FROM listings
    WHERE room_type = 'Entire home/apt'
),

cohort_metrics AS (
    SELECT
        experience_cohort,

        COUNT(*)                                                AS total_listings,

        -- Price metrics
        ROUND(AVG(price), 2)                                    AS avg_price,
        ROUND(AVG(price_per_bedroom), 2)                        AS avg_price_per_bedroom,

        -- Revenue metrics
        ROUND(AVG(estimated_revenue_l365d), 0)                  AS avg_annual_revenue,

        -- Occupancy
        ROUND(AVG(estimated_occupancy_l365d), 1)                AS avg_occupancy,

        -- Quality
        ROUND(AVG(review_scores_rating), 2)                     AS avg_rating,
        ROUND(AVG(reviews_per_month), 2)                        AS avg_reviews_per_month,

        -- Host profile
        ROUND(100.0 * SUM(host_is_superhost) / COUNT(*), 1)     AS superhost_rate,
        ROUND(AVG(amenity_count), 1)                            AS avg_amenity_count,
        ROUND(AVG(calculated_host_listings_count), 1)           AS avg_listings_count

    FROM experience_cohorts
    GROUP BY experience_cohort
),

cohort_growth AS (
    SELECT
        *,
        -- Revenue growth vs previous cohort
        ROUND(avg_annual_revenue - LAG(avg_annual_revenue)
            OVER (ORDER BY experience_cohort), 0)               AS revenue_vs_prev_cohort,

        -- Price growth vs previous cohort
        ROUND(avg_price - LAG(avg_price)
            OVER (ORDER BY experience_cohort), 2)               AS price_vs_prev_cohort,

        -- Occupancy growth vs previous cohort
        ROUND(avg_occupancy - LAG(avg_occupancy)
            OVER (ORDER BY experience_cohort), 1)               AS occupancy_vs_prev_cohort,

        -- Superhost growth vs previous cohort
        ROUND(superhost_rate - LAG(superhost_rate)
            OVER (ORDER BY experience_cohort), 1)               AS superhost_vs_prev_cohort,

        -- Revenue rank
        RANK() OVER (ORDER BY avg_annual_revenue DESC)          AS revenue_rank

    FROM cohort_metrics
)

SELECT
    experience_cohort,
    total_listings,
    avg_price,
    price_vs_prev_cohort,
    avg_annual_revenue,
    revenue_vs_prev_cohort,
    avg_occupancy,
    occupancy_vs_prev_cohort,
    avg_rating,
    superhost_rate,
    superhost_vs_prev_cohort,
    avg_amenity_count,
    avg_listings_count,
    revenue_rank
FROM cohort_growth
ORDER BY experience_cohort ASC
"""

# ─── Execute and display ───────────────────────────────────────────────────────
q8_results = pd.read_sql_query(query_8, conn)

print("QUERY 8 — HOST EXPERIENCE CURVE")
print("=" * 65)
print(f"  Segment : Entire home/apt only")
print(f"  Method  : Experience bucketed into 7 cohorts")
print(f"            LAG() tracks growth cohort over cohort")
print()
print(q8_results.to_string(index=False))

QUERY 8 — HOST EXPERIENCE CURVE
  Segment : Entire home/apt only
  Method  : Experience bucketed into 7 cohorts
            LAG() tracks growth cohort over cohort

experience_cohort  total_listings  avg_price  price_vs_prev_cohort  avg_annual_revenue  revenue_vs_prev_cohort  avg_occupancy  occupancy_vs_prev_cohort  avg_rating  superhost_rate  superhost_vs_prev_cohort  avg_amenity_count  avg_listings_count  revenue_rank
  1. Under 1 year             104     178.95                   NaN              2279.0                     NaN           21.3                       NaN        4.83             3.8                       NaN               30.5                 7.3             7
     2. 1-2 years             472     169.91                 -9.04             11766.0                  9487.0           69.0                      47.7        4.79            52.3                      48.5               45.1                 8.1             6
     3. 2-3 years             401     172.44               